# KITTI: drive-grouped split, packaged once

**Part 2, step 1: splitting the data into training, validation and test sets.** This notebook is run once, before
the experiment notebooks in `notebooks/`. It reads the raw KITTI object-detection training set (Kaggle mirror
`klemenko/kitti-dataset`), splits it **by drive** (whole recording sequences go to one split, so near-identical
neighbouring frames never end up on both sides), and writes a compact, ready-to-train copy to the notebook output.
That output is published as the Kaggle dataset
[`fizzazaitoonbsds2022/kitti-drive-grouped-split`](https://www.kaggle.com/datasets/fizzazaitoonbsds2022/kitti-drive-grouped-split),
so every experiment notebook starts from the identical split, and the test split can be downloaded on its own for the
app. A random image-level split is written as well, only for the leakage experiment (RQ3, notebook 06).

Runs on **CPU** (no GPU quota). About 15 minutes.

**Output** (`/kaggle/working/`):

| file | contents |
|---|---|
| `images_train.zip`, `images_val.zip`, `images_test.zip` | the PNG images of each split (stored, not compressed: PNG is already compressed) |
| `labels_yolo.zip` | YOLO labels `labels/{train,val,test}/<id>.txt` (8 classes, normalised `cx cy w h`) |
| `labels_raw.zip` | the original KITTI `label_2` files (all 15 fields, incl. DontCare) |
| `tables/` | `images.csv`, `annotations.csv`, `ignore_regions.csv`: one row per image / object / DontCare box, with both split columns |
| `splits/` | image-id lists `grouped_{train,val,test}.txt`, `random_{train,val,test}.txt`, `splits.csv`, `split_class_counts.csv` |
| `random_{train,val,test}.txt` | YOLO image lists for the random split (RQ3), pointing into the same image folders |
| `README.md` | dataset card (source, licence, layout) |

Zips instead of loose files because Kaggle handles a few large files much better than 15 000 small ones.

In [ ]:
import os, sys, json, shutil, zipfile, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

SEED = 42
CLASSES = ["Car", "Pedestrian", "Van", "Cyclist", "Truck", "Misc", "Tram", "Person_sitting"]   # Part 1 order
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
SPLITS = ["train", "val", "test"]
EXPECTED = {"train": 4699, "val": 1239, "test": 1543}     # grouped split sizes of Part 1 / Part 2 v1

ON_KAGGLE = Path("/kaggle/input").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path("out")
TMP = (Path("/kaggle/tmp") if Path("/kaggle/tmp").exists() else Path("/tmp")) if ON_KAGGLE else Path("tmp")
for d in (WORK, TMP):
    d.mkdir(parents=True, exist_ok=True)
print("output ->", WORK.resolve())

## 1. Find the raw data

The images and labels are copied from the mounted input to local disk first with many parallel readers: the input
mount is slow per file, local disk is fast for everything that follows.

In [ ]:
def find_dir(suffix, root="/kaggle/input", max_depth=6):
    skip = {"image_2", "label_2", "velodyne", "calib", "image_3"}
    base = len(Path(root).parts)
    for r, dirs, _ in os.walk(root):
        for d in dirs:
            if str(Path(r) / d).endswith(suffix):
                return Path(r) / d
        dirs[:] = [] if len(Path(r).parts) - base >= max_depth else [d for d in dirs if d not in skip]


def copy_dir(src, dst, workers=64):
    dst.mkdir(parents=True, exist_ok=True)
    entries = [e for e in os.scandir(src) if e.is_file()]
    cp = lambda e: (dst / e.name).exists() or shutil.copyfile(e.path, dst / e.name)
    with ThreadPoolExecutor(workers) as ex:
        list(tqdm(ex.map(cp, entries), total=len(entries), desc=f"copy {src.name}"))
    return dst


if ON_KAGGLE:
    src_img, src_lbl = find_dir("training/image_2"), find_dir("training/label_2")
    assert src_img and src_lbl, "attach the dataset klemenko/kitti-dataset"
    IMG_DIR = copy_dir(src_img, TMP / "image_2")
    LBL_DIR = copy_dir(src_lbl, TMP / "label_2")
else:                                  # local run: point these at an unzipped KITTI training folder
    IMG_DIR = Path(os.environ.get("KITTI_IMG_DIR", "data/raw/training/image_2"))
    LBL_DIR = Path(os.environ.get("KITTI_LBL_DIR", "data/raw/training/label_2"))
print(len(list(IMG_DIR.glob("*.png"))), "images |", len(list(LBL_DIR.glob("*.txt"))), "label files")

## 2. Parse images and labels

* image width / height (read from the PNG header only);
* every label line, keeping the 8 object classes; `DontCare` boxes are kept **separately** as ignore regions
  (the evaluator does not count detections inside them as false positives);
* the official KITTI difficulty of each object (Easy / Moderate / Hard, from box height, occlusion and truncation).

In [ ]:
COLS = ["type", "truncated", "occluded", "alpha", "x1", "y1", "x2", "y2",
        "h3d", "w3d", "l3d", "x3d", "y3d", "z3d", "rot_y"]


def image_size(p):
    with Image.open(p) as im:
        return {"image_id": p.stem, "width": im.size[0], "height": im.size[1]}


paths = sorted(IMG_DIR.glob("*.png"))
with ThreadPoolExecutor(32) as ex:
    images = pd.DataFrame(list(tqdm(ex.map(image_size, paths), total=len(paths), desc="image sizes")))

rows = []
for p in tqdm(sorted(LBL_DIR.glob("*.txt")), desc="label files"):
    for line in p.read_text().splitlines():
        parts = line.split()
        if len(parts) >= 15:
            rows.append([p.stem] + parts[:15])
raw = pd.DataFrame(rows, columns=["image_id"] + COLS)
num = [c for c in COLS if c != "type"]
raw[num] = raw[num].apply(pd.to_numeric, errors="coerce")
raw["occluded"] = raw["occluded"].astype(int)

ignore = raw[raw["type"] == "DontCare"].copy()
objs = raw[raw["type"].isin(CLASSES)].merge(images, on="image_id", how="left")
objs["bw"], objs["bh"] = objs.x2 - objs.x1, objs.y2 - objs.y1
objs["is_easy"] = (objs.bh >= 40) & (objs.occluded <= 0) & (objs.truncated <= 0.15)
objs["is_moderate"] = (objs.bh >= 25) & (objs.occluded <= 1) & (objs.truncated <= 0.30)
objs["is_hard"] = (objs.bh >= 25) & (objs.occluded <= 2) & (objs.truncated <= 0.50)
objs["difficulty"] = np.select([objs.is_easy, objs.is_moderate, objs.is_hard],
                               ["Easy", "Moderate", "Hard"], default="Unrated")
objs["class_id"] = objs["type"].map(CLASS_TO_ID)
objs = objs.reset_index(drop=True)
print(f"{len(images)} images | {len(objs)} objects | {len(ignore)} DontCare regions")
objs["type"].value_counts()

## 3. Which drive does each image come from?

KITTI's training images are frames from a few dozen recording drives, shuffled. The official devkit's
`train_mapping.txt` and `train_rand.txt` map every image back to its drive. The devkit is small and is downloaded
from the KITTI server if the attached dataset does not contain it.

In [ ]:
DEVKIT_URL = "https://s3.eu-central-1.amazonaws.com/avg-kitti/devkit_object.zip"


def find_devkit_file(name):
    roots = [Path("/kaggle/input"), TMP / "devkit"] if ON_KAGGLE else [Path("data/raw"), TMP / "devkit"]
    for r in roots:
        hits = list(r.rglob(name)) if r.exists() else []
        if hits:
            return hits[0]
    dst = TMP / "devkit"
    dst.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(DEVKIT_URL, dst / "devkit_object.zip")
    zipfile.ZipFile(dst / "devkit_object.zip").extractall(dst)
    return next(dst.rglob(name))


mapping = pd.read_csv(find_devkit_file("train_mapping.txt"), sep=r"\s+", header=None, names=["date", "drive", "raw_frame"])
rand = [int(x) for x in find_devkit_file("train_rand.txt").read_text().replace("\n", "").split(",") if x.strip()]
assert len(mapping) == len(rand) == len(images)
m = mapping.iloc[[rand[i] - 1 for i in images["image_id"].astype(int)]].reset_index(drop=True)
images["drive"] = m["drive"].values
images["raw_frame"] = m["raw_frame"].astype(int).values
print(images.drive.nunique(), "drives")

## 4. The splits

**Drive-grouped split (used for everything).** Whole drives go to train / validation / test (70 / 15 / 15 %).
Drives are visited from largest to smallest (ties shuffled with seed 42) and each goes to the split whose image
count and per-class object counts move closest to their targets. Neighbouring frames of one drive are nearly
identical, so this is what keeps the test set honest.

**Random image-level split (RQ3 only).** The same images, shuffled per image with seed 42. Kept only to measure how
much such a split overstates performance.

Same algorithm and seed as Part 1, so the split is identical; the assertions below check that.

In [ ]:
def grouped_split(images, objs, ratios=(0.70, 0.15, 0.15), seed=SEED):
    feats = pd.DataFrame({"n_images": images.groupby("drive").size()})
    cnt = objs.merge(images[["image_id", "drive"]], on="image_id").groupby(["drive", "type"]).size().unstack(fill_value=0)
    for c in CLASSES:
        feats[c] = cnt[c] if c in cnt else 0
    feats = feats.fillna(0).astype(float)
    target = {s: feats.sum() * r for s, r in zip(SPLITS, ratios)}
    cur = {s: pd.Series(0.0, index=feats.columns) for s in SPLITS}
    cost = lambda v, t: (((v - t) / (t + 1e-9)) ** 2).sum()
    order = feats.sample(frac=1, random_state=seed).sort_values("n_images", ascending=False, kind="stable").index
    assign = {}
    for g in order:
        deltas = {s: cost(cur[s] + feats.loc[g], target[s]) - cost(cur[s], target[s]) for s in SPLITS}
        assign[g] = min(deltas, key=deltas.get)
        cur[assign[g]] = cur[assign[g]] + feats.loc[g]
    return assign


def random_split(n, seed=SEED):
    perm = np.random.default_rng(seed).permutation(n)
    lab = np.empty(n, dtype=object)
    lab[perm[: int(.70 * n)]], lab[perm[int(.70 * n): int(.85 * n)]], lab[perm[int(.85 * n):]] = "train", "val", "test"
    return lab


images["split_grouped"] = images["drive"].map(grouped_split(images, objs))
images["split_random"] = random_split(len(images))

# checks: no drive in two splits, no image in two splits, same sizes as before
assert (images.groupby("drive")["split_grouped"].nunique() == 1).all(), "a drive appears in two splits"
assert images.image_id.is_unique
sizes = images.split_grouped.value_counts().to_dict()
assert sizes == EXPECTED, f"split differs from Part 1: {sizes}"
print("grouped:", sizes, "| random:", images.split_random.value_counts().to_dict())

counts = []
for kind in ["grouped", "random"]:
    o = objs.merge(images[["image_id", f"split_{kind}"]], on="image_id")
    t = pd.crosstab(o[f"split_{kind}"], o["type"]).reindex(index=SPLITS, columns=CLASSES, fill_value=0)
    t.insert(0, "drives", images.groupby(f"split_{kind}")["drive"].nunique())
    t.insert(0, "images", images[f"split_{kind}"].value_counts())
    counts.append(t.assign(split_kind=kind).reset_index(names="split"))
counts = pd.concat(counts, ignore_index=True)
counts

## 5. Write the tables, split lists and YOLO labels

YOLO label line: `class_id cx cy w h`, normalised to 0-1, boxes clipped to the image. The random split reuses the
same image folders through list files (`./images/<folder>/<id>.png`), so no image is stored twice.

In [ ]:
(WORK / "tables").mkdir(exist_ok=True)
(WORK / "splits").mkdir(exist_ok=True)
images.to_csv(WORK / "tables" / "images.csv", index=False)
objs.merge(images[["image_id", "drive", "split_grouped", "split_random"]], on="image_id") \
    .to_csv(WORK / "tables" / "annotations.csv", index=False)
ignore.merge(images[["image_id", "split_grouped", "split_random"]], on="image_id")[
    ["image_id", "x1", "y1", "x2", "y2", "split_grouped", "split_random"]].to_csv(WORK / "tables" / "ignore_regions.csv", index=False)

for kind in ["grouped", "random"]:
    for s in SPLITS:
        ids = images.loc[images[f"split_{kind}"] == s, "image_id"]
        (WORK / "splits" / f"{kind}_{s}.txt").write_text("\n".join(ids) + "\n")
images[["image_id", "drive", "raw_frame", "split_grouped", "split_random"]].to_csv(WORK / "splits" / "splits.csv", index=False)
counts.to_csv(WORK / "splits" / "split_class_counts.csv", index=False)

folder = dict(zip(images.image_id, images.split_grouped))      # image -> its folder (grouped split)
for s in SPLITS:
    ids = images.loc[images.split_random == s, "image_id"]
    (WORK / f"random_{s}.txt").write_text("\n".join(f"./images/{folder[i]}/{i}.png" for i in ids) + "\n")


def yolo_line(b):
    x1, y1, x2, y2 = max(b.x1, 0), max(b.y1, 0), min(b.x2, b.width), min(b.y2, b.height)
    if x2 <= x1 or y2 <= y1:
        return None
    return f"{b.class_id} {(x1 + x2) / 2 / b.width:.6f} {(y1 + y2) / 2 / b.height:.6f} {(x2 - x1) / b.width:.6f} {(y2 - y1) / b.height:.6f}"


by_img = dict(tuple(objs.groupby("image_id")))
with zipfile.ZipFile(WORK / "labels_yolo.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for r in tqdm(images.itertuples(), total=len(images), desc="YOLO labels"):
        lines = [l for l in map(yolo_line, by_img[r.image_id].itertuples()) if l] if r.image_id in by_img else []
        z.writestr(f"labels/{r.split_grouped}/{r.image_id}.txt", "\n".join(lines))
with zipfile.ZipFile(WORK / "labels_raw.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in tqdm(sorted(LBL_DIR.glob("*.txt")), desc="raw labels"):
        z.write(p, f"label_2/{p.name}")

## 6. Package the images

One zip per split, `images/<split>/<id>.png` inside. Unzipping all three gives the standard YOLO folder layout
next to `labels/`.

In [ ]:
for s in SPLITS:
    ids = images.loc[images.split_grouped == s, "image_id"]
    with zipfile.ZipFile(WORK / f"images_{s}.zip", "w", zipfile.ZIP_STORED) as z:
        for i in tqdm(ids, desc=f"images_{s}.zip"):
            z.write(IMG_DIR / f"{i}.png", f"images/{s}/{i}.png")

## 7. Dataset card and final check

In [ ]:
card = f'''# KITTI 2D object detection: drive-grouped split (Group 16)

Re-packaged from the KITTI Vision Benchmark object-detection training set (7,481 images), via `klemenko/kitti-dataset`.

**Licence / credit:** KITTI is published under CC BY-NC-SA 3.0. Geiger, Lenz, Urtasun, "Are we ready for Autonomous
Driving? The KITTI Vision Benchmark Suite", CVPR 2012. http://www.cvlibs.net/datasets/kitti/

**Split:** whole recording drives assigned to train / val / test (70/15/15, greedy balance of images and the 8 classes,
seed 42): {sizes}. `random_*.txt` = image-level random split, only for measuring leakage.

**Layout:** unzip `images_*.zip` and `labels_yolo.zip` into one folder -> `images/<split>/`, `labels/<split>/` (YOLO).
Classes (id order): {CLASSES}. `tables/` holds per-image / per-object / DontCare tables with both split columns.
'''
(WORK / "README.md").write_text(card)

for p in sorted(WORK.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(WORK)!s:45} {p.stat().st_size / 1e6:9.1f} MB")
for s in SPLITS:
    with zipfile.ZipFile(WORK / f"images_{s}.zip") as z:
        assert len(z.namelist()) == EXPECTED[s], s
print("all checks passed")

## 8. Sanity check: one training image with its labels

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

iid = images.loc[images.split_grouped == "train", "image_id"].iloc[0]
img = Image.open(IMG_DIR / f"{iid}.png")
fig, ax = plt.subplots(figsize=(14, 4.5))
ax.imshow(img)
for b in objs[objs.image_id == iid].itertuples():
    ax.add_patch(patches.Rectangle((b.x1, b.y1), b.bw, b.bh, fill=False, lw=1.5, color="lime"))
    ax.text(b.x1, b.y1 - 3, f"{b.type} ({b.difficulty})", color="lime", fontsize=8)
for b in ignore[ignore.image_id == iid].itertuples():
    ax.add_patch(patches.Rectangle((b.x1, b.y1), b.x2 - b.x1, b.y2 - b.y1, fill=False, lw=1, ls="--", color="red"))
ax.set_title(f"{iid}: green = labelled objects, red dashed = DontCare (ignored)")
ax.axis("off")
plt.show()

# free the local copy so only the outputs remain
if ON_KAGGLE:
    shutil.rmtree(TMP / "image_2", ignore_errors=True)